# Phase A1 — Path C+ 6-node minimal fix (Oracle retrain, 3 seeds × 25 epochs)

**Goal** : produce 3-seed retrain results for thesis defence (H1 hypothesis).

**Code path** : `scripts.finetune_stage1_bundle_b.finetune_bundle_b` (Bundle B warm-start FT, NOT `train_epoch_stage1`).

**Strategy** :
- Warm-start each seed from `V5_DIR/epoch_last.pth` (PC1 fresh checkpoint protocol).
- 25 epochs Bundle B FT per seed × 3 seeds = [42, 7, 123].
- Output to `ORACLE_DIR/seed_<n>/` (per user request : `oracle` naming = V5 model alias in thesis).
- A100 expected runtime : ~6-10h/seed × 3 = ~20-30h sequential. Colab Pro+ session limit 24h → user resumes after each session.
- Per-seed JSON tagged with `stamp_batch_d_json` → PC4 audit-gate eligible.
- Cross-seed aggregation : BCa CI + Student-t CI (PC8 sd floor 0.05) + Cohen's d.

**Two PASS bars reported (PC6 caveat clause)** :
- **STRICT H1 (PC5)** : Q_phys_cont ≥ 0.50 mean over 3 seeds + BCa+Student-t CI lower bounds > 0.04 baseline + Cohen's d ≥ 0.8.
- **M2 DEFENDABLE** : Q_phys_cont ≥ 0.30 with significance vs baseline (interventional sign-consistency, structural caveat per PC6).

**Resume semantics** : the notebook checks if `ORACLE_DIR/seed_<n>/results.json` already exists, and skips that seed. To re-run a seed, delete the corresponding directory.

In [ ]:
# >>> COLAB_BOOTSTRAP (same as smoke #4 but with A1 commit verification)
import os, sys, subprocess, time, shlex
from pathlib import Path

GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "four-node-causal"
LOCAL_PROJECT = "/content/climate_data"
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()


def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    print(f"$ {cmd}")
    t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0
    print(f"  rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc


if _IS_COLAB:
    _T0 = time.time()
    print("Colab détecté — bootstrap A1…\n")

    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        project_path.parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")

    if GIT_PULL_ON_RESUME:
        print("=" * 70)
        print(f"GIT SYNC — force {LOCAL_PROJECT} to origin/{GIT_BRANCH}")
        print("=" * 70)
        try:
            current_branch = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --abbrev-ref HEAD")
            ).decode().strip()
        except Exception:
            current_branch = "(unknown)"
        print(f"   Current branch : {current_branch}")

        _run(f"git -C {LOCAL_PROJECT} fetch --depth=200 origin {GIT_BRANCH}",
             timeout=180, check=True)
        if current_branch != GIT_BRANCH:
            _run(f"git -C {LOCAL_PROJECT} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}",
                 timeout=30, check=True)
        else:
            _run(f"git -C {LOCAL_PROJECT} reset --hard origin/{GIT_BRANCH}",
                 timeout=30, check=True)

        try:
            head_sha = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --short HEAD")
            ).decode().strip()
            head_msg = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} log -1 --pretty=%s")
            ).decode().strip()
            print(f"   HEAD = {head_sha}  ({head_msg})")
            print(f"   Last 8 commits on {GIT_BRANCH}:")
            log_out = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} log -8 --oneline")
            ).decode().strip()
            for _line in log_out.splitlines():
                print(f"     {_line}")
        except Exception as e:
            print(f"   (HEAD verification failed: {e})")
        print()

    os.chdir(project_path)
    print(f"   chdir → {os.getcwd()}\n")

    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    _need_pip = True
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            _need_pip = False
            print("OK Imports critiques — pip install sauté.")
        except ImportError as _imp_err:
            print(f"   import st_cdgm a échoué ({_imp_err}) — pip install requis.")

    if _need_pip:
        EXTRA_DEPS = [
            "omegaconf==2.3.0", "hydra-core==1.3.2",
            "diffusers==0.36.0", "transformers==4.57.6",
            "accelerate==1.12.0", "huggingface-hub==0.36.0",
            "safetensors==0.7.0", "xbatcher", "webdataset",
            "cftime", "h5netcdf", "numcodecs",
            "torch-geometric", "xformers",
        ]
        deps_str = " ".join(shlex.quote(p) for p in EXTRA_DEPS)
        _run(f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location {deps_str}",
             timeout=600)
        _run(f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
             f"--no-deps -e {LOCAL_PROJECT}", timeout=120)

    print(f"\nBootstrap A1 terminé en {time.time() - _T0:.1f}s.")
else:
    _here = Path.cwd()
    for _candidate in [_here, *_here.parents]:
        if (_candidate / "config" / "training_config.yaml").exists() and (_candidate / "setup.py").exists():
            if _candidate != _here:
                os.chdir(_candidate)
            break
    print("Hors Colab — bootstrap sauté.")

print()
print("=" * 70)
print("Path C+ A1 Verifications")
print("=" * 70)
try:
    import inspect as _ins
    from scripts.finetune_stage1_bundle_b import DEFAULT_HYPERPARAMS as _DH, finetune_bundle_b  # noqa
    if _DH.get("dag_gate_warmup_start_epoch") is None:
        print("OK Batch F-1 (DEFAULT_HYPERPARAMS gate=None -> auto-scale)")
    print(f"OK finetune_bundle_b imported")
    print(f"   lr_encoder = {_DH.get('lr_encoder')}")
    print(f"   lr_rcn     = {_DH.get('lr_rcn')}")
    print(f"   lambda_l1_start = {_DH.get('lambda_l1_start')}")
    print(f"   lambda_l1_end   = {_DH.get('lambda_l1_end')}")
    print(f"   lambda_dag_prior = {_DH.get('lambda_dag_prior')}")
    print(f"   g_phys_alpha     = {_DH.get('g_phys_alpha')}")
except Exception as e:
    print(f"FAIL A1 verification: {e}")
print()


In [ ]:
# === Cell 2 : GPU profile + config + ORACLE_DIR setup ===
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner
from omegaconf import OmegaConf

GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# A1 requires A100 (per AI eng + math prof bandwidth analysis)
if GPU_PROFILE.get("profile_id") not in ("a100", "h100"):
    print()
    print("WARN A1 was budgeted for A100. Detected:", GPU_PROFILE.get("profile_id"))
    print("   T4/V100 will multiply wall-time by 5-8x. Consider Pro+ A100 access.")

CONFIG = OmegaConf.load("config/training_config.yaml")
_corrdiff = OmegaConf.load("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

# Apply GPU profile
CONFIG.training.batch_size = GPU_PROFILE["batch_size"]
CONFIG.training.use_amp = GPU_PROFILE["use_amp"]
CONFIG.training.num_workers = GPU_PROFILE["num_workers"]

from pathlib import Path
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === ORACLE_DIR (per user spec) ===
# V5 = Oracle (causal model name in thesis, per project_model_naming.md memory).
# Each seed gets its own subdirectory : oracle/seed_42/, seed_7/, seed_123/
# The V5_DIR baseline (fresh checkpoint) is the FT starting point per PC1.
V5_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal")
ORACLE_DIR = Path("/content/drive/MyDrive/climate_data/oracle")
ORACLE_DIR.mkdir(parents=True, exist_ok=True)
print(f"[A1] V5_DIR    = {V5_DIR}  (warm-start source, NOT modified)")
print(f"[A1] ORACLE_DIR = {ORACLE_DIR}  (A1 retrain outputs)")
print(f"[A1] Existing seed dirs in oracle/: {sorted(p.name for p in ORACLE_DIR.iterdir() if p.is_dir())}")

# Verify V5_DIR ckpt exists (PC4 audit gate prerequisite)
_v5_ckpt = V5_DIR / "epoch_last.pth"
if not _v5_ckpt.exists():
    raise FileNotFoundError(
        f"V5_DIR/epoch_last.pth not found at {_v5_ckpt}. "
        f"A1 cannot proceed without warm-start baseline."
    )
print(f"[A1] V5_DIR ckpt OK : {_v5_ckpt.stat().st_size/1024**3:.2f} GB")


In [ ]:
# === Cell 3 : Build pipeline + builder + sample dataset (re-usable across seeds) ===
import itertools as _it
import numpy as np
from torch.utils.data import Dataset as _TorchDataset

from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    GraphToGridDecoder, RCNCell, RCNSequenceRunner,
    CausalDiffusionDecoder,
)
from st_cdgm.models.edm_preconditioner import EDMConfig
try:
    from st_cdgm.models import ConditionalSkipBlock
    SKIP_AVAILABLE = True
except ImportError:
    SKIP_AVAILABLE = False
    ConditionalSkipBlock = None

# Data root (same as smoke)
DATA_ROOT = Path("/content/drive/MyDrive/climate_data/data")
LR_PATH = str(DATA_ROOT / "train/predictor_ACCESS-CM2_hist.nc")
HR_PATH = str(DATA_ROOT / "train/pr_ACCESS-CM2_hist.nc")
STATIC_PATH = str(DATA_ROOT / "static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc")
MEAN_PATH = str(DATA_ROOT / "normalization_coefs/mean_1974_2011.nc")
STD_PATH = str(DATA_ROOT / "normalization_coefs/std_1974_2011.nc")

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=int(CONFIG.data.seq_len),
    baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    target_transform=str(CONFIG.data.get("target_transform", "log1p")),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.get("precipitation_delta", 0.01)),
    lr_variables=list(CONFIG.data.lr_variables),
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables),
    means_path=MEAN_PATH, stds_path=STD_PATH,
    eager_load_datasets=False,
)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=bool(CONFIG.graph.include_mid_layer),
)
print(f"OK Builder created: {len(builder.dynamic_node_types)} dyn + {len(builder.static_node_types)} static nodes")


class _MapStyleListDataset(_TorchDataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        return self.samples[i]


# === A1 dataset sizing (larger than smoke 100, smaller than full) ===
# Smoke #4 used 100 train samples. Full ACCESS-CM2 1980-2011 = ~11000 time steps.
# For A1 we want enough data to differentiate from smoke but stay within A100 budget.
# 1000 train + 200 val = ~10% of full dataset. At A100 bs=16:
#   - 1000/16 = 62 batches/epoch × 25 epochs = 1550 batches/seed
#   - ~3-5h/seed × 3 seeds = ~10-15h total wallclock (within 1 Colab Pro+ session)
A1_TRAIN_SAMPLES = 1000
A1_VAL_SAMPLES = 200

print(f"[A1] Materializing {A1_TRAIN_SAMPLES} train + {A1_VAL_SAMPLES} val samples...")
train_iter = pipeline.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len), stride=1, as_torch=True,
)
_train_samples = list(_it.islice(train_iter, A1_TRAIN_SAMPLES))
train_dataset = _MapStyleListDataset(_train_samples)
print(f"  [OK] train_dataset : {len(train_dataset)} samples")

val_iter = pipeline.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len),
    stride=int(CONFIG.data.stride),
    as_torch=True,
)
_val_samples = list(_it.islice(val_iter, A1_VAL_SAMPLES))
val_dataset = _MapStyleListDataset(_val_samples)
print(f"  [OK] val_dataset   : {len(val_dataset)} samples")

# Detect runtime driver_dim
_runtime_dim = int(_train_samples[0]["lr"].shape[1])
if _runtime_dim != int(CONFIG.rcn.driver_dim):
    CONFIG.rcn.driver_dim = _runtime_dim
    CONFIG.rcn.reconstruction_dim = _runtime_dim


def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {nt: lr_nodes_steps[0] for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {"lr": lr_tensor, "residual": sample["residual"],
            "baseline": sample.get("baseline"), "hetero": hetero}


# encoder_configs (shared across seeds; encoder is rebuilt per seed)
allowed_nodes = set(builder.dynamic_node_types + builder.static_node_types)
encoder_configs = []
for _mp in CONFIG.encoder.metapaths:
    _src, _rel, _tgt = _mp.src, _mp.relation, _mp.target
    if _src in allowed_nodes and _tgt in allowed_nodes:
        encoder_configs.append(IntelligibleVariableConfig(
            name=_mp.name,
            meta_path=(_src, _rel, _tgt),
            pool=_mp.get("pool", "mean"),
        ))
if pipeline.get_static_dataset() is not None:
    encoder_configs.append(IntelligibleVariableConfig(
        name="static", meta_path=("SP_HR", "causes", "GP850"), pool="mean",
    ))
num_vars = len(encoder_configs)
hr_channels = int(_train_samples[0]["residual"].shape[1])

print(f"[A1] num_vars = {num_vars}, hr_channels = {hr_channels}")
print(f"[A1] Estimated wall-time per seed (A100, bs={GPU_PROFILE.get('batch_size')}) :")
_batches_per_epoch = A1_TRAIN_SAMPLES // GPU_PROFILE.get("batch_size", 16)
_total_batches = _batches_per_epoch * 25
print(f"        {_batches_per_epoch} batches/epoch × 25 epochs = {_total_batches} batches")
print(f"        @ ~2 batches/sec on A100 = ~{_total_batches/2/60:.1f} min = ~{_total_batches/2/3600:.1f} h")


In [ ]:
# === Cell 4 : Q_phys metrics + projection hook factory (re-usable across seeds) ===
import warnings
import json
import shutil as _sh
from scripts.finetune_stage1_bundle_b import finetune_bundle_b
from src.st_cdgm.training.physics_prior import build_physical_mask, VAR_LABELS


def compute_q_phys_binary(A_dag_np, G_phys_np, threshold=0.01):
    A = np.array(A_dag_np); np.fill_diagonal(A, 0.0); G = np.array(G_phys_np)
    mask = G != 0
    if not mask.any(): return 0.0, 0, 0, 0
    A_thresh = A.copy(); A_thresh[np.abs(A_thresh) <= threshold] = 0.0
    matches = int(((np.sign(A_thresh) == np.sign(G)) & mask).sum())
    n_phys = int(mask.sum())
    n_extra = int(((G == 0) & (np.abs(A) > threshold) & (~np.eye(A.shape[0], dtype=bool))).sum())
    return matches / n_phys, matches, n_phys, n_extra


def compute_q_phys_adaptive(A_dag_np, G_phys_np, frac=0.3):
    A = np.array(A_dag_np); np.fill_diagonal(A, 0.0); G = np.array(G_phys_np)
    threshold = max(0.01, frac * float(np.abs(A).max()))
    mask = G != 0
    if not mask.any(): return 0.0, 0, 0, threshold
    A_thresh = A.copy(); A_thresh[np.abs(A_thresh) <= threshold] = 0.0
    matches = int(((np.sign(A_thresh) == np.sign(G)) & mask).sum())
    return matches / int(mask.sum()), matches, int(mask.sum()), threshold


def compute_q_phys_continuous(A_dag_np, G_phys_np, *, collapse_eps=1e-12):
    A = np.array(A_dag_np); np.fill_diagonal(A, 0.0); G = np.array(G_phys_np)
    mask_phys = G != 0
    sign_correct = (np.sign(A) == np.sign(G)) & mask_phys
    num = float(np.abs(A[sign_correct]).sum())
    den = float(np.abs(A).sum())
    if den < collapse_eps: return 0.0, True
    return num / den, False


def compute_phys_mag_gained(A_dag_now, A_dag_init, G_phys_np):
    A_now = np.array(A_dag_now); A_init = np.array(A_dag_init); G = np.array(G_phys_np)
    np.fill_diagonal(A_now, 0.0); np.fill_diagonal(A_init, 0.0)
    mask_phys = G != 0
    sc_now = (np.sign(A_now) == np.sign(G)) & mask_phys
    sc_init = (np.sign(A_init) == np.sign(G)) & mask_phys
    return float(np.abs(A_now[sc_now]).sum() - np.abs(A_init[sc_init]).sum())


def compute_skeleton_f1(A_dag_np, G_phys_np, frac=0.3):
    A = np.array(A_dag_np); np.fill_diagonal(A, 0.0); G = np.array(G_phys_np)
    threshold = max(0.01, frac * float(np.abs(A).max()))
    A_skel = ((np.abs(A) > threshold) | (np.abs(A.T) > threshold)).astype(int)
    np.fill_diagonal(A_skel, 0)
    G_skel = ((G != 0) | (G.T != 0)).astype(int)
    np.fill_diagonal(G_skel, 0)
    iu = np.triu_indices_from(A_skel, k=1)
    A_e = A_skel[iu]; G_e = G_skel[iu]
    tp = int(((A_e == 1) & (G_e == 1)).sum())
    fp = int(((A_e == 1) & (G_e == 0)).sum())
    fn = int(((A_e == 0) & (G_e == 1)).sum())
    if tp + fp == 0 or tp + fn == 0: return 0.0, threshold
    p = tp / (tp + fp); r = tp / (tp + fn)
    if p + r == 0: return 0.0, threshold
    return 2 * p * r / (p + r), threshold


G_phys = build_physical_mask(num_vars=num_vars)
G_phys_np = G_phys.numpy()
print(f"[A1] G_phys built : shape {G_phys_np.shape}, sum {int(G_phys_np.sum())} physical edges")


def install_projection_hook(rcn_cell):
    """3-point projection hook (pre_spectral / post_spectral / post_floor)."""
    log = {
        "pre_spectral_A_dag": [],
        "post_spectral_A_dag": [],
        "post_floor_A_dag": [],
        "spectral_rescale": [],
        "floor_rescale": [],
    }
    _orig_spectral = rcn_cell.project_dag_spectral
    _orig_floor = rcn_cell.project_dag_floor

    def _patched_spectral(max_radius=0.95):
        pre = rcn_cell.A_dag.data.detach().cpu().clone().numpy()
        rescale = _orig_spectral(max_radius=max_radius)
        post = rcn_cell.A_dag.data.detach().cpu().clone().numpy()
        log["pre_spectral_A_dag"].append(pre)
        log["post_spectral_A_dag"].append(post)
        log["spectral_rescale"].append(float(rescale))
        return rescale

    def _patched_floor(min_norm=0.10, prior=None):
        rescale = _orig_floor(min_norm=min_norm, prior=prior)
        post = rcn_cell.A_dag.data.detach().cpu().clone().numpy()
        log["post_floor_A_dag"].append(post)
        log["floor_rescale"].append(float(rescale))
        return rescale

    rcn_cell.project_dag_spectral = _patched_spectral
    rcn_cell.project_dag_floor = _patched_floor
    return log


print("[A1] Helpers ready : Q_phys variants + projection hook + skeleton F1")


In [ ]:
# === Cell 5 : Per-seed training function (resume-aware) ===
A1_EPOCHS = 25
SEEDS = [42, 7, 123]


def build_fresh_stack(seed):
    """Build a fresh stack from V5_DIR ckpt (per PC1 fresh checkpoint protocol).
    
    Each seed gets its own stack from the SAME V5_DIR/epoch_last.pth -- no
    cross-seed contamination.
    """
    enc = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=CONFIG.encoder.hidden_dim,
        conditioning_dim=CONFIG.encoder.conditioning_dim,
    ).to(DEVICE)

    rcn_cell = RCNCell(
        num_vars=num_vars,
        hidden_dim=CONFIG.rcn.hidden_dim,
        driver_dim=int(CONFIG.rcn.driver_dim),
        reconstruction_dim=int(CONFIG.rcn.reconstruction_dim),
        dropout=CONFIG.rcn.dropout,
    ).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

    rh = GraphToGridDecoder(
        d_model=CONFIG.encoder.hidden_dim,
        hr_h=CONFIG.graph.hr_shape[0], hr_w=CONFIG.graph.hr_shape[1],
    ).to(DEVICE)

    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
    _unet_kwargs = OmegaConf.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ("down_block_types", "up_block_types"):
        if _k in _unet_kwargs and isinstance(_unet_kwargs[_k], list):
            _unet_kwargs[_k] = tuple(_unet_kwargs[_k])
    diff = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        unet_kwargs=_unet_kwargs,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=bool(CONFIG.diffusion.get("use_gradient_checkpointing", False)),
        conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
        anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
        edm_config=edm_cfg,
        causal_concat=True,
    ).to(DEVICE)

    skip = None
    if SKIP_AVAILABLE:
        skip = ConditionalSkipBlock(
            lr_channels=len(CONFIG.data.lr_variables),
            hr_shape=tuple(CONFIG.graph.hr_shape),
        ).to(DEVICE)

    # === Load V5_DIR ckpt FRESH (PC1) ===
    ckpt = torch.load(V5_DIR / "epoch_last.pth", map_location=DEVICE, weights_only=False)
    load_audit = {
        "encoder_state_dict_n_keys": len(ckpt.get("encoder_state_dict", {})),
        "modules_loaded_strict": [],
        "modules_loaded_fallback": [],
        "skip_block_loaded_from_ckpt": False,
    }
    assert load_audit["encoder_state_dict_n_keys"] > 0, "DS audit: encoder_state_dict empty"

    def _safe_load(name_, module, sd):
        try:
            module.load_state_dict(sd, strict=True)
            load_audit["modules_loaded_strict"].append(name_)
            return True
        except RuntimeError:
            module.load_state_dict(sd, strict=False)
            load_audit["modules_loaded_fallback"].append(name_)
            return False

    _safe_load("encoder", enc, ckpt.get("encoder_state_dict", {}))
    _safe_load("rcn_cell", rcn_cell, ckpt.get("rcn_cell_state_dict", {}))
    _safe_load("regression_head", rh, ckpt.get("regression_head_state_dict", {}))
    _safe_load("diffusion", diff, ckpt.get("diffusion_state_dict", {}))
    if skip is not None and ckpt.get("skip_block_state_dict") is not None:
        _safe_load("skip_block", skip, ckpt["skip_block_state_dict"])
        load_audit["skip_block_loaded_from_ckpt"] = True

    A_dag = rcn_cell.A_dag.detach().cpu().clone()
    stack = {
        "encoder": enc, "rcn_runner": rcn_runner, "regression_head": rh,
        "diffusion": diff, "skip_block": skip, "A_dag": A_dag,
        "variant": f"Oracle-A1-seed-{seed}",
    }
    return stack, rcn_cell, load_audit


def run_a1_seed(seed):
    """Run A1 for a single seed. Returns the results dict.
    
    Resume semantics : if ORACLE_DIR/seed_<seed>/results.json exists, skip
    and return its contents.
    """
    seed_dir = ORACLE_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    results_path = seed_dir / "results.json"

    if results_path.exists():
        print(f"\n[A1 seed {seed}] Results already exist at {results_path}, skipping.")
        return json.loads(results_path.read_text())

    print(f"\n{'='*72}")
    print(f"[A1 seed {seed}] STARTING (V5_DIR fresh -> {A1_EPOCHS} epochs FT)")
    print(f"{'='*72}")
    _t_start = time.time()

    stack, rcn_cell, load_audit = build_fresh_stack(seed)
    projection_log = install_projection_hook(rcn_cell)

    A_dag_initial = rcn_cell.A_dag.detach().cpu().numpy()
    # Initial metrics
    q_bin_init, m_init, n_phys, n_extra_init = compute_q_phys_binary(A_dag_initial, G_phys_np)
    q_adapt_init, m_adapt_init, _, t_adapt_init = compute_q_phys_adaptive(A_dag_initial, G_phys_np)
    q_cont_init, _ = compute_q_phys_continuous(A_dag_initial, G_phys_np)
    skel_init, skel_t_init = compute_skeleton_f1(A_dag_initial, G_phys_np)

    print(f"  Q_phys binary init   : {q_bin_init:.4f} ({m_init}/{n_phys})")
    print(f"  Q_phys adaptive init : {q_adapt_init:.4f} (thresh={t_adapt_init:.4f})")
    print(f"  Q_phys continuous    : {q_cont_init:.4f}")
    print(f"  Skeleton F1 init     : {skel_init:.4f}")
    print(f"  A_dag norm init      : {(A_dag_initial**2).sum()**0.5:.4f}")

    # Train
    with warnings.catch_warnings(record=True) as w_record:
        warnings.simplefilter("always")
        result = finetune_bundle_b(
            stack=stack,
            builder=builder,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            CONFIG=CONFIG,
            DEVICE=DEVICE,
            epochs=A1_EPOCHS,
            batch_size=GPU_PROFILE["batch_size"],
            ckpt_save_dir=seed_dir,
            convert_sample_to_batch_fn=convert_sample_to_batch,
            sanity_eval_every=5,
            seed=seed,
            skip_sigma_data_recalib=False,
        )

    # Final metrics
    A_dag_final = rcn_cell.A_dag.detach().cpu().numpy()
    A_dag_pre_spectral = projection_log["pre_spectral_A_dag"][-1].copy() if projection_log["pre_spectral_A_dag"] else A_dag_final.copy()

    q_bin_final, m_final, _, n_extra_final = compute_q_phys_binary(A_dag_final, G_phys_np)
    q_adapt_final, m_adapt_final, _, t_adapt_final = compute_q_phys_adaptive(A_dag_final, G_phys_np)
    q_cont_final, collapsed_final = compute_q_phys_continuous(A_dag_final, G_phys_np)
    skel_final, skel_t_final = compute_skeleton_f1(A_dag_final, G_phys_np)
    phys_mag_gained = compute_phys_mag_gained(A_dag_final, A_dag_initial, G_phys_np)

    j8_failed = [w for w in w_record if "J8 skip_block forward failed" in str(w.message)]
    missing_edge = [w for w in w_record if "§1.6 missing-edge fallback" in str(w.message)]
    n_spec_fired = sum(1 for r in projection_log["spectral_rescale"] if abs(r - 1.0) > 1e-9)
    n_floor_fired = sum(1 for r in projection_log["floor_rescale"] if abs(r - 1.0) > 1e-9)

    elapsed = time.time() - _t_start
    print(f"\n  Q_phys binary final  : {q_bin_final:.4f} ({m_final}/{n_phys})")
    print(f"  Q_phys adaptive final: {q_adapt_final:.4f} (thresh={t_adapt_final:.4f})")
    print(f"  Q_phys continuous    : {q_cont_final:.4f}")
    print(f"  Phys mag gained      : {phys_mag_gained:+.4f}")
    print(f"  A_dag norm final     : {(A_dag_final**2).sum()**0.5:.4f}")
    print(f"  Skeleton F1 final    : {skel_final:.4f}")
    print(f"  Elapsed              : {elapsed/60:.1f} min ({elapsed/3600:.2f}h)")

    results = {
        "seed": seed,
        "a1_epochs": A1_EPOCHS,
        "a1_train_samples": A1_TRAIN_SAMPLES,
        "a1_val_samples": A1_VAL_SAMPLES,
        "gpu_profile": GPU_PROFILE.get("profile_id"),
        "commit_sha": subprocess.check_output(shlex.split("git rev-parse HEAD")).decode().strip() if _IS_COLAB else None,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "elapsed_sec": float(elapsed),
        "code_path_scope": {
            "training_fn": "scripts.finetune_stage1_bundle_b.finetune_bundle_b",
            "warm_start_from": str(V5_DIR / "epoch_last.pth"),
            "output_dir": str(seed_dir),
        },
        # Q_phys variants
        "q_phys_binary_initial": float(q_bin_init),
        "q_phys_binary_final": float(q_bin_final),
        "q_phys_adaptive_initial": float(q_adapt_init),
        "q_phys_adaptive_final": float(q_adapt_final),
        "q_phys_adaptive_threshold_final": float(t_adapt_final),
        "q_phys_continuous_initial": float(q_cont_init),
        "q_phys_continuous_final": float(q_cont_final),
        "q_phys_continuous_collapsed_final": bool(collapsed_final),
        "phys_mag_gained": phys_mag_gained,
        # Counts
        "q_phys_matches_initial": m_init,
        "q_phys_matches_final": m_final,
        "q_phys_n_physical_edges": n_phys,
        "n_extra_edges_initial": n_extra_init,
        "n_extra_edges_final": n_extra_final,
        # A_dag matrices
        "a_dag_initial_matrix": A_dag_initial.tolist(),
        "a_dag_final_matrix": A_dag_final.tolist(),
        "a_dag_pre_spectral_matrix": A_dag_pre_spectral.tolist(),
        "a_dag_norm_initial": float((A_dag_initial**2).sum()**0.5),
        "a_dag_norm_final": float((A_dag_final**2).sum()**0.5),
        "a_dag_asymmetry_initial": float(((A_dag_initial - A_dag_initial.T)**2).sum()**0.5),
        "a_dag_asymmetry_final": float(((A_dag_final - A_dag_final.T)**2).sum()**0.5),
        "g_phys_matrix": G_phys_np.tolist(),
        # Skeleton + warnings
        "skeleton_f1_initial": float(skel_init),
        "skeleton_f1_final": float(skel_final),
        "j8_failures": len(j8_failed),
        "missing_edge_fallbacks": len(missing_edge),
        # Projector activity
        "projector_spectral_fired_count": int(n_spec_fired),
        "projector_spectral_total_calls": len(projection_log["spectral_rescale"]),
        "projector_floor_fired_count": int(n_floor_fired),
        "projector_floor_total_calls": len(projection_log["floor_rescale"]),
        # Audit (PC4 gate)
        "load_audit": load_audit,
    }

    # Apply Batch-D stamp (PC4-eligible, valid_for_analysis=True)
    try:
        from path_c_plus.scripts._tombstone_legacy_jsons import stamp_batch_d_json
        results = stamp_batch_d_json(
            results,
            j29_scheduler_type=str(CONFIG.diffusion.scheduler_type),
            j29_cfg_scale=1.0,
            pre_registration_commit=results["commit_sha"],
        )
    except Exception as e:
        print(f"  [warn] stamp_batch_d_json failed: {e}")

    results_path.write_text(json.dumps(results, indent=2, default=str))
    print(f"  Saved -> {results_path}")
    return results


print(f"[A1] Ready to run {len(SEEDS)} seeds : {SEEDS}")
print(f"     Output -> {ORACLE_DIR}/seed_<n>/results.json")
print(f"     Resume : existing JSONs are skipped (delete to re-run)")


In [ ]:
# === Cell 6 : Run all 3 seeds sequentially (resume-aware) ===
# On Colab Pro+ A100, each seed ~6-10h. 24h session limit -> 1-2 seeds per session.
# Resume by re-running this cell : seeds with existing results.json are skipped.
#
all_results = []
for seed in SEEDS:
    try:
        result = run_a1_seed(seed)
        all_results.append(result)
    except Exception as e:
        print(f"\n[A1 seed {seed}] FAILED with {type(e).__name__}: {e}")
        print(f"  Re-run this cell after fixing to retry.")
        raise

print(f"\n{'='*72}")
print(f"[A1] All {len(all_results)} seeds completed")
print(f"{'='*72}")
for r in all_results:
    print(f"  seed {r['seed']:3d} : Q_phys_cont = {r['q_phys_continuous_final']:.4f}, "
          f"adaptive = {r['q_phys_adaptive_final']:.4f}, "
          f"binary = {r['q_phys_binary_final']:.4f}, "
          f"n_extra = {r['n_extra_edges_final']}")


In [ ]:
# === Cell 7 : Cross-seed PC8 statistical analysis (BCa + Student-t + Cohen's d) ===
import scipy.stats as _stats
from path_c_plus.scripts.stats_utils import (
    bootstrap_ci_3seeds,
    paired_wilcoxon_oracle_vs_corrdiff,
    holm_bonferroni_correction,
)

q_cont_per_seed = [r["q_phys_continuous_final"] for r in all_results]
q_adapt_per_seed = [r["q_phys_adaptive_final"] for r in all_results]
phys_mag_per_seed = [r["phys_mag_gained"] for r in all_results]
n_extra_per_seed = [r["n_extra_edges_final"] for r in all_results]

baseline_cont = 0.04   # V5-mini baseline (band-diagonal dominated)
random_null = 0.083    # PC8 amendment #5 : E[Q_phys_cont | A iid Normal] = 0.5 * 5/30
h1_threshold = 0.50    # PC5 H1 bar
m2_threshold = 0.30    # PC6 partial-defendable bar

mean_cont = float(np.mean(q_cont_per_seed))
sd_cont = float(np.std(q_cont_per_seed, ddof=1))  # sample SD (n-1)
# PC8 sd floor 0.05 to prevent d=infinity on near-collapse
sd_cont_floored = max(sd_cont, 0.05)

# Cohen's d (one-sample standardized mean vs baseline 0.04)
cohens_d = (mean_cont - baseline_cont) / sd_cont_floored

# Student-t one-sided CI lower bound (df=2)
t_crit = _stats.t.ppf(0.95, df=2)  # one-sided 95%
student_t_lower = mean_cont - t_crit * sd_cont_floored / (3 ** 0.5)

# BCa CI (1000 resamples)
bca_result = bootstrap_ci_3seeds(q_cont_per_seed, n_resamples=1000,
                                   confidence=0.95, method="bca", rng_seed=42)
bca_lower = bca_result["ci_lower"]
bca_upper = bca_result["ci_upper"]

print("=" * 72)
print("A1 CROSS-SEED ANALYSIS (PC5/PC6/PC8)")
print("=" * 72)
print(f"\nQ_phys_continuous per seed : {[f'{x:.4f}' for x in q_cont_per_seed]}")
print(f"  mean           = {mean_cont:.4f}")
print(f"  std (n-1)      = {sd_cont:.4f}")
print(f"  std floored    = {sd_cont_floored:.4f} (PC8 floor 0.05)")
print(f"  Cohen's d      = {cohens_d:.4f} (vs baseline 0.04)")
print(f"  BCa 95% CI     = [{bca_lower:.4f}, {bca_upper:.4f}]")
print(f"  Student-t 95%  = lower {student_t_lower:.4f}")
print()
print(f"PC5 STRICT H1 bar :")
print(f"  mean >= 0.50           : {('OK' if mean_cont >= h1_threshold else 'KO')}  ({mean_cont:.4f})")
print(f"  BCa lower > 0.04       : {('OK' if bca_lower > baseline_cont else 'KO')}  ({bca_lower:.4f})")
print(f"  Student-t lower > 0.04 : {('OK' if student_t_lower > baseline_cont else 'KO')}  ({student_t_lower:.4f})")
print(f"  BCa lower > 0.083 (rnd): {('OK' if bca_lower > random_null else 'KO')}  ({bca_lower:.4f})")
print(f"  Cohen's d >= 0.8       : {('OK' if cohens_d >= 0.8 else 'KO')}  ({cohens_d:.4f})")
print()
print(f"PC6 M2 DEFENDABLE bar :")
print(f"  mean >= 0.30           : {('OK' if mean_cont >= m2_threshold else 'KO')}  ({mean_cont:.4f})")
print(f"  BCa lower > 0.04       : {('OK' if bca_lower > baseline_cont else 'KO')}  ({bca_lower:.4f})")
print(f"  n_extra_edges < 3      : {('OK' if max(n_extra_per_seed) < 3 else 'CAVEAT')}  ({n_extra_per_seed})")
print()

# Verdict
h1_pass = (mean_cont >= h1_threshold
           and bca_lower > baseline_cont
           and student_t_lower > baseline_cont
           and bca_lower > random_null
           and cohens_d >= 0.8)
m2_pass = mean_cont >= m2_threshold and bca_lower > baseline_cont

if h1_pass:
    A1_VERDICT = "H1_PASS"
    msg = "H1 ACCEPTED. Path C+ Q_phys_cont >= 0.50 with CI lower > baseline upper."
elif m2_pass and max(n_extra_per_seed) < 3:
    A1_VERDICT = "M2_PASS"
    msg = "M2 thesis defendable. Q_phys_cont >= 0.30 with significance, sparse structural recovery."
elif m2_pass:
    A1_VERDICT = "M2_PARTIAL_PC6"
    msg = "M2 PARTIAL per PC6 : interventional sign-consistency achieved, sparse structural recovery NOT achieved."
else:
    A1_VERDICT = "FAIL"
    msg = "A1 FAILED both bars. Investigate per PC3 (gradient diagnostics) before Path C+."

print(f"VERDICT : {A1_VERDICT}")
print(f"        {msg}")

# Save aggregate
aggregate = {
    "verdict": A1_VERDICT,
    "verdict_message": msg,
    "q_phys_continuous_per_seed": q_cont_per_seed,
    "q_phys_adaptive_per_seed": q_adapt_per_seed,
    "phys_mag_gained_per_seed": phys_mag_per_seed,
    "n_extra_edges_per_seed": n_extra_per_seed,
    "mean_q_phys_continuous": mean_cont,
    "sd_q_phys_continuous": sd_cont,
    "sd_q_phys_continuous_floored": sd_cont_floored,
    "cohens_d_vs_baseline": cohens_d,
    "bca_ci_95": [bca_lower, bca_upper],
    "student_t_lower_95": student_t_lower,
    "baseline_cont": baseline_cont,
    "random_null": random_null,
    "h1_threshold": h1_threshold,
    "m2_threshold": m2_threshold,
    "h1_pass": h1_pass,
    "m2_pass": m2_pass,
    "n_seeds": len(all_results),
    "seeds": SEEDS,
    "commit_sha": all_results[0]["commit_sha"] if all_results else None,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "per_seed_results_paths": [str(ORACLE_DIR / f"seed_{s}" / "results.json") for s in SEEDS],
}

# Stamp aggregate as Batch-D eligible
try:
    from path_c_plus.scripts._tombstone_legacy_jsons import stamp_batch_d_json
    aggregate = stamp_batch_d_json(
        aggregate,
        j29_scheduler_type=str(CONFIG.diffusion.scheduler_type),
        j29_cfg_scale=1.0,
        pre_registration_commit=aggregate["commit_sha"],
    )
except Exception as e:
    print(f"[warn] stamp_batch_d_json on aggregate failed: {e}")

agg_path = ORACLE_DIR / "a1_aggregate.json"
agg_path.write_text(json.dumps(aggregate, indent=2, default=str))
print(f"\n[A1] Aggregate saved -> {agg_path}")
print(f"[A1] schema_version  = {aggregate.get('schema_version')}")
print(f"[A1] valid_for_analysis = {aggregate.get('valid_for_analysis')}")
print(f"[A1] Push oracle/a1_aggregate.json to GitHub for team review")
